# Clean and Process df

## Set Up

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH, SEED
from src.utils import col_config_og, col_config_sensitivity
import pandas as pd

pd.set_option("display.max_columns", 500)
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv(
    BASE_PATH / "data" / "complete_raw" / "comppsych_dawbas_20210414-0.csv"
)
df.shape

In [ ]:
self_impact_cols = col_config_og["self_impact"]
parent_impact_cols = col_config_og["parent_impact"]
self_text_cols = col_config_og["self_text"]
parent_text_cols = col_config_og["parent_text"]
self_quest_cols = col_config_og["self_quest"]
parent_quest_cols = col_config_og["parent_quest"]
meta_cols = col_config_og["meta"]

## Clean

Remove duplicated SDAN (23717.0) at idx 234 (double the amount of null), same patient

In [ ]:
df_clean = df.drop(index=234)

Remove participants who `removed consent`

In [ ]:
df_clean = df_clean[df_clean["Eligible"] != "Did not or withdrew assent/consent"]

Only keep participants who were designated as `Participant_Type2` MDD (remove HV, NaN, Anxious, Unsure)

In [ ]:
df_clean = df_clean[df_clean["Participant_Type2"] == "MDD"]
df_clean.shape

Create new eligibility col to stratify on excluded bc meets criteria, excluded bc not meet criteria, & everything else

In [ ]:
mapping = {
    "Excluded: meets exclusionary criteria (substance use, psychosis, etc.)": "excluded_meets_criteria",
    "Excluded: does not meet criteria": "excluded_no_meet_criteria",
    "Excluded: cannot be reached or scheduled, all contact options exhausted": "Include",
    "Include: can't scan (braces, etc.)": "Include",
}

df_clean["eligible_group"] = df_clean["Eligible"].map(mapping).fillna("Include")
df_clean["eligible_group"].value_counts()

Subset to only include relevant depression + general cols

In [ ]:
all_self_cols = self_quest_cols + self_text_cols + self_impact_cols
all_parent_cols = parent_quest_cols + parent_text_cols + parent_impact_cols
all_depr_cols = all_self_cols + all_parent_cols
keep_cols = meta_cols + all_depr_cols
df_subset = df_clean[keep_cols]
df_subset.shape

Split

In [ ]:
from sklearn.model_selection import train_test_split
from shutil import rmtree

# export raw
export_dir = BASE_PATH / "data" / "cleaned" / "initial_split"
if export_dir.exists():
    rmtree(export_dir)
export_dir.mkdir(exist_ok=False, parents=True)
df_subset.to_parquet(export_dir / "raw_clean_depression.parquet")

# split
X_train, X_test, _, _ = train_test_split(
    df_subset,
    np.ones(len(df_subset)),
    train_size=0.33,
    random_state=SEED,
    shuffle=True,
    stratify=df_subset["eligible_group"],
)
X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
## Export
X_train.to_parquet(export_dir / "train.parquet")
X_test.to_parquet(export_dir / "test.parquet")

## Work only on train set

In [ ]:
df_sub = pd.read_parquet(
    BASE_PATH / "data" / "cleaned" / "initial_split" / "train.parquet"
)
df_sub.shape

Seperate self & parent dfs

In [ ]:
self_df = df_sub[meta_cols + self_quest_cols + self_text_cols + self_impact_cols]
parent_df = df_sub[
    meta_cols + parent_quest_cols + parent_text_cols + parent_impact_cols
]

## Fill/drop NAs


- Create cols for Depression. Irritability, Loss of Interest checks
- Used to skip questions later
- Skipped questions due to these^ will be marked as 0 (questionairre) or "Not Applicable" (text)
- Non-structural NAs should be dropped?

### Self

Create cols

In [ ]:
# Depr --> H1, H2, AND H3
self_df["Depression_Check_self"] = (
    (self_df["s_dep_1_sad"] == 1)
    & (self_df["s_dep_2_miserable_daily"] == 1)
    & (self_df["s_dep_3_miserable_mostly"] == 1)
)
# Irr --> H7, H8, AND H9
self_df["Irritable_Check_self"] = (
    (self_df["s_dep_7_irritable"] == 1)
    & (self_df["s_dep_8_irritable_daily"] == 1)
    & (self_df["s_dep_9_irratable_mostly"] == 1)
)
# Loss of Interest --> H13 AND H14
self_df["Loss_Interest_Check_self"] = (self_df["s_dep_13_anhedonia"] == 1) & (
    self_df["s_dep_14_anhedonia_daily"] == 1
)

Mark NA as 0 for skipped questions (bc of answer to a previous question)

In [ ]:
## ====> Miserable (2-5); For 2-5, change NAs to 0s when H1 is 0
skipped_after_1 = [
    "s_dep_2_miserable_daily",
    "s_dep_3_miserable_mostly",
    "s_dep_4_miserable_cheered_up",
    "s_dep_5_miserable_duration",
]
mask = self_df["s_dep_1_sad"] == 0
self_df.loc[mask, skipped_after_1] = self_df.loc[mask, skipped_after_1].fillna(0)
## ====> Irritiable (8-11); For 8-11, change NAs to 0s when H7 is 0
skipped_after_7 = [
    "s_dep_8_irritable_daily",
    "s_dep_9_irratable_mostly",
    "s_dep_10_irritable_improved",
    "s_dep_11_irritable_duration",
]
mask = self_df["s_dep_7_irritable"] == 0
self_df.loc[mask, skipped_after_7] = self_df.loc[mask, skipped_after_7].fillna(0)
## ====> anhedonia (14-16); for 14-16, change NAs to 0s when H13 is 0
skipped_after_13 = [
    "s_dep_14_anhedonia_daily",
    "s_dep_15_anhedonia_mostly",
    "s_dep_16_anhedonia_duration",
]
mask = self_df["s_dep_13_anhedonia"] == 0
self_df.loc[mask, skipped_after_13] = self_df.loc[mask, skipped_after_13].fillna(0)
## ====> H17; Answer only if H13 AND (Depr OR Irr); Change NA to 0 if conditition not met
mask = (self_df["s_dep_13_anhedonia"] == 0) | (
    (self_df["Depression_Check_self"] == 0) & (self_df["Irritable_Check_self"] == 0)
)
self_df.loc[mask, "s_dep_17_coincided_mis_irrit"] = self_df.loc[
    mask, "s_dep_17_coincided_mis_irrit"
].fillna(0)
## ====> Depr + Irr + LoI (18A-18L, 19, 20A-20D, 21)
# Only need 1 to be True to answer
skipped_at_18 = [
    "s_dep_18a_tired",
    "s_dep_18b_appetite",
    "s_dep_18c_weight",
    "s_dep_18d_insomnia",
    "s_dep_18e_hypersomnia",
    "s_dep_18f_agitation",
    "s_dep_18g_worthless",
    "s_dep_18h_concentration",
    "s_dep_18i_thoughts_death",
    "s_dep_18j_self_harm_talk_recent",
    "s_dep_18k_self_harm_act_recent",
    "s_dep_18l_self_harm_act_ever",
] + self_impact_cols
# These also only answered if one of Depr/Irr/LoI
skipped_text = self_text_cols
mask = (
    (self_df["Depression_Check_self"] == 0)
    & (self_df["Irritable_Check_self"] == 0)
    & (self_df["Loss_Interest_Check_self"] == 0)
)
# Fill questions with 0
self_df.loc[mask, skipped_at_18] = self_df.loc[mask, skipped_at_18].fillna(0)
# Fill NA text with Not Applicable
self_df.loc[mask, skipped_text] = self_df.loc[mask, skipped_text].fillna(
    "Not applicable"
)

Remove NAs resulting from true missingness rather than structure

- DAWBA_ID 230319 (idx 42); 14 pure NAs, and most other entries are 0
- DAWBA_ID 230368 (idx 73); no response after 18; meaning no impact score
- DAWBA_IDs: [234579, 230092, 234994, 234219, 234433, 230385, 235017, 230310, 234329] (indices: [3, 11, 13, 15, 17, 23, 34, 48, 69])
	- mostly good but no answers for any text
- DAWBA_ID: 234774 (idx 9); only answers for 2 text questions, the rest NA

In [ ]:
drop_indices = [42, 73, 3, 11, 13, 15, 17, 23, 34, 48, 69, 9]
self_df = self_df.drop(drop_indices, axis=0)
self_df.shape

Remove death/self-harm questions bc of missingness (44 in each of 18i, 18j, 18k,16l)

In [ ]:
drop_cols = [
    "s_dep_18i_thoughts_death",
    "s_dep_18j_self_harm_talk_recent",
    "s_dep_18k_self_harm_act_recent",
    "s_dep_18l_self_harm_act_ever",
]
self_quest_cols = [col for col in self_quest_cols if col not in drop_cols]
self_df = self_df.drop(drop_cols, axis=1)

One entry only missing 2 text entries, fill with "Not answered" (DAWBA_ID 234493)

- Instead of removing entire participant

In [ ]:
fill_cols = ["s_dep_25g_text_ep_past", "s_dep_25h_text_ep_high"]
self_df[fill_cols] = self_df[fill_cols].fillna("Not answered")

In [ ]:
assert self_df.isna().sum().sum() == 0

### Parent

Create cols

In [ ]:
# Depr --> H1, H2, AND H3
parent_df["Depression_Check_parent"] = (
    (parent_df["p_dep_1_sad"] == 1)
    & (parent_df["p_dep_2_miserable_daily"] == 1)
    & (parent_df["p_dep_3_miserable_mostly"] == 1)
)
# Irr --> H7, H8, AND H9
parent_df["Irritable_Check_parent"] = (
    (parent_df["p_dep_7_irritable"] == 1)
    & (parent_df["p_dep_8_irritable_daily"] == 1)
    & (parent_df["p_dep_9_irratable_mostly"] == 1)
)
# Loss of Interest --> H13 AND H14
parent_df["Loss_Interest_Check_parent"] = (parent_df["p_dep_13_anhedonia"] == 1) & (
    parent_df["p_dep_14_anhedonia_daily"] == 1
)

Mark NA as 0 for skipped questions

In [ ]:
## ====> Miserable (2-5); For 2-5, change NAs to 0s when H1 is 0
skipped_after_1 = [
    "p_dep_2_miserable_daily",
    "p_dep_3_miserable_mostly",
    "p_dep_4_miserable_cheered_up",
    "p_dep_5_miserable_duration",
]
mask = parent_df["p_dep_1_sad"] == 0
parent_df.loc[mask, skipped_after_1] = parent_df.loc[mask, skipped_after_1].fillna(0)
## ====> Irritiable (8-11); For 8-11, change NAs to 0s when H7 is 0
skipped_after_7 = [
    "p_dep_8_irritable_daily",
    "p_dep_9_irratable_mostly",
    "p_dep_10_irritable_improved",
    "p_dep_11_irritable_duration",
]
mask = parent_df["p_dep_7_irritable"] == 0
parent_df.loc[mask, skipped_after_7] = parent_df.loc[mask, skipped_after_7].fillna(0)
## ====> anhedonia (14-16); for 14-16, change NAs to 0s when H13 is 0
skipped_after_13 = [
    "p_dep_14_anhedonia_daily",
    "p_dep_15_anhedonia_mostly",
    "p_dep_16_anhedonia_duration",
]
mask = parent_df["p_dep_13_anhedonia"] == 0
parent_df.loc[mask, skipped_after_13] = parent_df.loc[mask, skipped_after_13].fillna(0)
## ====> H17; Answer only if H13 AND (Depr OR Irr); Change NA to 0 if conditition not met
mask = (parent_df["p_dep_13_anhedonia"] == 0) | (
    (parent_df["Depression_Check_parent"] == 0)
    & (parent_df["Irritable_Check_parent"] == 0)
)
parent_df.loc[mask, "p_dep_17_coincided_mis_irrit"] = parent_df.loc[
    mask, "p_dep_17_coincided_mis_irrit"
].fillna(0)
## ====> Depr + Irr + LoI (18A-18L, 19, 20A-20D, 21)
# Only need 1 to be True to answer
skipped_at_18 = [
    "p_dep_18a_tired",
    "p_dep_18b_appetite",
    "p_dep_18c_weight",
    "p_dep_18d_insomnia",
    "p_dep_18e_hypersomnia",
    "p_dep_18f_agitation",
    "p_dep_18g_worthless",
    "p_dep_18h_concentration",
    "p_dep_18i_thoughts_death",
    "p_dep_18j_self_harm_talk_recent",
    "p_dep_18k_self_harm_act_recent",
    "p_dep_18l_self_harm_act_ever",
] + parent_impact_cols
# These also only answered if one of Depr/Irr/LoI
skipped_text = parent_text_cols
mask = (
    (parent_df["Depression_Check_parent"] == 0)
    & (parent_df["Irritable_Check_parent"] == 0)
    & (parent_df["Loss_Interest_Check_parent"] == 0)
)
# Fill questions with 0
parent_df.loc[mask, skipped_at_18] = parent_df.loc[mask, skipped_at_18].fillna(0)
# Fill NA text with Not Applicable
parent_df.loc[mask, skipped_text] = parent_df.loc[mask, skipped_text].fillna(
    "Not applicable"
)

Remove death/self-harm questions bc of missingness (40 in each of 18i, 18j, 18k,16l)

In [ ]:
drop_cols = [
    "p_dep_18i_thoughts_death",
    "p_dep_18j_self_harm_talk_recent",
    "p_dep_18k_self_harm_act_recent",
    "p_dep_18l_self_harm_act_ever",
]
parent_quest_cols = [col for col in parent_quest_cols if col not in drop_cols]
parent_df = parent_df.drop(drop_cols, axis=1)

Remove NAs resulting from true missingness rather than structure

- DAWBA_ID 234331 (idx 14): 14 pure NAs, most other entries 0
- DAWBA_IDs [235007, 234440, 234996, 230212, 234989, 234766] (idxs [10, 18, 47, 57, 59, 64])
	- Missing entries for 7,8,9,10,11 --> maybe answered no for 7 and rest got marked NA?
	- remove for now
- DAWBA_IDs [234579, 230058, 230385, 234736, 234965, 230339, 230151, 234488, 234457] (idxs [3, 21, 23, 32, 36, 39, 41, 43, 54])
	- Missing most of text cols 

-  DAWBA_IDs 230088, 234219 (idxs 8, 15); missing 4-5/9 text entries
	- Could add "Not answered" to keep these participants?
	- remove for now

In [ ]:
drop_indices = [14, 10, 18, 47, 57, 59, 64, 3, 21, 23, 32, 36, 39, 41, 43, 54, 8, 15]
parent_df = parent_df.drop(drop_indices, axis=0)
parent_df.shape

One entry missing 2 text entries, fill with "Not answered" (DAWBA ID: 234466, idx 67)

In [ ]:
fill_cols = ["p_dep_25h_text_ep_high", "p_dep_25j_text_tried_fix"]
parent_df[fill_cols] = parent_df[fill_cols].fillna("Not answered")

In [ ]:
assert parent_df.isna().sum().sum() == 0

## Export

Create impact cols

In [ ]:
## Self
self_df["impact_score_self"] = self_df[self_impact_cols].sum(axis=1)
## Parent
parent_df["impact_score_parent"] = parent_df[parent_impact_cols].sum(axis=1)

Export base dfs

In [ ]:
export_dir = BASE_PATH / "data" / "cleaned" / "processed"
if export_dir.exists():
    rmtree(export_dir)
export_dir.mkdir(exist_ok=False, parents=True)
# Self
self_df = self_df.reset_index(drop=True)
self_df.to_parquet(export_dir / "self.parquet")
# Parent
parent_df = parent_df.reset_index(drop=True)
parent_df.to_parquet(export_dir / "parent.parquet")

# Also process meta info file for easier use later

In [ ]:
from shutil import rmtree

new_self_text_cols = col_config_sensitivity["self_text"]
new_parent_text_cols = col_config_sensitivity["parent_text"]
meta_df = pd.read_excel(
    BASE_PATH / "data" / "complete_raw" / "dawba_column_names_and_descriptions.xlsx"
)

In [ ]:
keep_cols = ["new_name", "description"]
self_meta = meta_df[meta_df["new_name"].isin(new_self_text_cols)][keep_cols]
parent_meta = meta_df[meta_df["new_name"].isin(new_parent_text_cols)][keep_cols]
## Ensure we got exactly right number of columns
assert len(self_meta) == len(new_self_text_cols)
assert len(parent_meta) == len(new_parent_text_cols)

In [ ]:
def strip_prefix(meta_df):
    new_rows_list = []
    for idx, row in meta_df.iterrows():

        og_text = row.description
        og_text_split = og_text.split(")")
        # Get only text after `Dep: text n)` prefix + remove leading/trailing whitespace + remove new lines + remove double spaces
        new_text = (
            " ".join(og_text_split[1:]).strip().replace("\n", " ").replace("  ", " ")
        )
        cur_row = {"question_col": row.new_name, "question_text": new_text}
        new_rows_list.append(cur_row)
    return pd.DataFrame(new_rows_list)

In [ ]:
self_meta_cleaned = strip_prefix(self_meta)
parent_meta_cleaned = strip_prefix(parent_meta)

Export

In [ ]:
meta_export_dir = BASE_PATH / "data" / "cleaned" / "meta_text"
if meta_export_dir.exists():
    rmtree(meta_export_dir)
meta_export_dir.mkdir(exist_ok=False, parents=True)
self_meta_cleaned.to_csv(meta_export_dir / "self.csv")
parent_meta_cleaned.to_csv(meta_export_dir / "parent.csv")